In [84]:
from pyspark.sql.functions import *
from pyspark.sql import Window

StatementMeta(, df2a9ee5-29b5-4635-b891-06719edc6650, 262, Finished, Available, Finished, False)

In [85]:
def read_table(table_name):
    return spark.read.table(table_name)

StatementMeta(, df2a9ee5-29b5-4635-b891-06719edc6650, 263, Finished, Available, Finished, False)

In [86]:
def delete_unuseful_columns(df):
    return df.select('TIME_PERIOD', 'OBS_VALUE')

StatementMeta(, df2a9ee5-29b5-4635-b891-06719edc6650, 264, Finished, Available, Finished, False)

In [87]:
def get_min_max_values(df, column):
    return df.agg(
        min(column).alias("min_value"),
        max(column).alias("max_value")
    ).first()

StatementMeta(, df2a9ee5-29b5-4635-b891-06719edc6650, 265, Finished, Available, Finished, False)

In [88]:
def create_date_range(df, column):
    min_max = get_min_max_values(df, column)
    return spark.range(1).select(
        explode(
            sequence(
                lit(min_max["min_value"]),
                lit(min_max["max_value"]),
                lit(1).cast("interval day")
            )
        ).alias(column)
    )

StatementMeta(, df2a9ee5-29b5-4635-b891-06719edc6650, 266, Finished, Available, Finished, False)

In [89]:
def join_date_range(date_df, df):
    return date_df.join(df, date_df["TIME_PERIOD"] == df["TIME_PERIOD"], "left").drop(df["TIME_PERIOD"])

StatementMeta(, df2a9ee5-29b5-4635-b891-06719edc6650, 267, Finished, Available, Finished, False)

In [90]:
def fill_in_holiday_data(df, column):
    window = Window.orderBy("TIME_PERIOD")
    return df.withColumn(column, last(column, ignorenulls=True).over(window))

StatementMeta(, df2a9ee5-29b5-4635-b891-06719edc6650, 268, Finished, Available, Finished, False)

In [91]:
def rename_column(df, old_name, new_name):
    return df.withColumnRenamed(old_name, new_name)

StatementMeta(, df2a9ee5-29b5-4635-b891-06719edc6650, 269, Finished, Available, Finished, False)

In [92]:
def add_reverse_exchange_rate(df):
    return df.withColumn("usd_to_eur_multiplier", 1 / col("eur_to_usd_multiplier"))

StatementMeta(, df2a9ee5-29b5-4635-b891-06719edc6650, 270, Finished, Available, Finished, False)

In [93]:
def transform_exchange_rates(table_name):
    df = read_table(table_name)
    df = delete_unuseful_columns(df)
    date_df = create_date_range(df, "TIME_PERIOD")
    df = join_date_range(date_df, df)
    df = fill_in_holiday_data(df, "OBS_VALUE")
    df = rename_column(df, "TIME_PERIOD", "date")
    df = rename_column(df, "OBS_VALUE", "eur_to_usd_multiplier")
    return add_reverse_exchange_rate(df)

StatementMeta(, df2a9ee5-29b5-4635-b891-06719edc6650, 271, Finished, Available, Finished, False)

In [94]:
def save_table(df, table_name):
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)

StatementMeta(, df2a9ee5-29b5-4635-b891-06719edc6650, 272, Finished, Available, Finished, False)

In [95]:
df = transform_exchange_rates("Bronze_lakehouse.dbo.USD_EUR_FX_rates")
display(df)
save_table(df, "Silver_lakehouse.dbo.USD_EUR_FX_rates")

StatementMeta(, df2a9ee5-29b5-4635-b891-06719edc6650, 273, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 190b68ca-5eaa-4707-bffd-b1d46d6fa6d0)